# 🎭 Face-Swap & Deepfake Detection — Fast 10-Minute Training
### Training DINOv2 on Real Photos vs. Blended Face-Swaps & Synthetic Faces

This notebook trains **DINOv2-ViT-B/14** and **EfficientNet-B0** to catch **face-swaps** (boundary seam blending) as well as **generative AI faces**.
Pre-extracted crops are used, so **no slow MTCNN extraction** is required.

In [ ]:
# ── Cell 1: GPU Diagnostics ──────────────────────────────────────
import torch
print('=' * 60)
print('  GPU Diagnostics')
print('=' * 60)
print('CUDA Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device     :', torch.cuda.get_device_name(0))
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: Enable GPU accelerator in Settings (top-right)')


In [ ]:
%%time
# ── Cell 2: Dependencies & Project Setup ─────────────────────────
!pip install -q timm albumentations grad-cam tabulate seaborn pyyaml

import os
REPO_URL = 'https://github.com/mamunur77/Deepfake.git'
PROJECT_DIR = '/kaggle/working/deepfake-vit'

if os.path.exists(PROJECT_DIR):
    print(f'[setup] Updating repository at {PROJECT_DIR}...')
    !cd {PROJECT_DIR} && git pull origin main
else:
    print(f'[setup] Cloning repository from {REPO_URL}...')
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /kaggle/working/deepfake-vit
print('[setup] Current working directory:', os.getcwd())


In [ ]:
%%time
# ── Cell 3: Build Balanced Hybrid Face-Swap Splits ───────────────
!python src/data/build_ai_face_splits.py


In [ ]:
%%time
# ── Cell 4: Train DINOv2 & EfficientNet (5 Epochs, Batch 64) ──────
import os, yaml

configs = ['configs/dinov2_partial.yaml', 'configs/efficientnet_b0.yaml']

for cfg_path in configs:
    with open(cfg_path, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['training']['epochs'] = 5
    cfg['training']['batch_size'] = 64
    cfg['training']['early_stopping_patience'] = 3
    with open(cfg_path, 'w') as f:
        yaml.safe_dump(cfg, f)

for cfg_path in configs:
    print('\n' + '=' * 70)
    print(f'  Training: {cfg_path}')
    print('=' * 70)
    exit_code = os.system(f'python scripts/train.py --config {cfg_path}')
    if exit_code != 0:
        print(f'  [WARNING] Training failed for {cfg_path}')


In [ ]:
%%time
# ── Cell 5: Evaluation & Comparison ──────────────────────────────
!python scripts/evaluate.py --checkpoint outputs/dinov2_partial/best_model.pt --split-csv data/splits/test.csv
!python scripts/evaluate.py --checkpoint outputs/efficientnet_b0/best_model.pt --split-csv data/splits/test.csv
!python scripts/compare_runs.py


In [ ]:
# ── Cell 6: Zip Outputs & Download Link ──────────────────────────
import os
from IPython.display import FileLink

!zip -r /kaggle/working/deepfake_face_swaps.zip outputs/ -x '*.tfevents*'

%cd /kaggle/working
print('\n' + '=' * 60)
print('  Click below to download your Face-Swap detector models:')
print('=' * 60)
display(FileLink('deepfake_face_swaps.zip'))
